# S1 Staging ? Budget

Standardize the budget numeric types and validate the unique month-shop grain. The month remains its source `YYYY-MM` string and amounts remain in CZK.


# 1. Imports & Configuration


In [ ]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "raw_data").is_dir() and (path / "s1_staging").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the FABINI project root from the current directory.")

RAW_DIR = PROJECT_ROOT / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "s1_staging" / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_FILE = "raw_budget.csv"
OUTPUT_PATH = OUTPUT_DIR / "budget.parquet"
EXPECTED_ROWS = 27
EXPECTED_SHOPS = ["CZ", "SK", "HU"]
EXPECTED_COLUMNS = ["month", "shop", "orders_budget", "net_sales_czk_budget"]


# 2. Load Raw Data

Read all source values as text to avoid implicit type inference.


In [ ]:
raw = pl.read_csv(RAW_DIR / SOURCE_FILE, infer_schema=False, null_values=[""])
if raw.columns != EXPECTED_COLUMNS or raw.height != EXPECTED_ROWS:
    raise ValueError(f"Unexpected budget source shape: {raw.height} rows, columns={raw.columns}")
missing_required = {column: raw[column].null_count() for column in EXPECTED_COLUMNS if raw[column].null_count()}
if missing_required:
    raise ValueError(f"Required budget values are NULL: {missing_required}")
if raw.filter(~pl.col("month").str.contains(r"^\d{4}-(0[1-9]|1[0-2])$")).height:
    raise ValueError("Budget month must use YYYY-MM format")
display(raw.head(5))


# 3. Cleaning & Typing

Keep `month` as its verified `YYYY-MM` representation; cast order targets to Int64 and CZK sales targets to Decimal(18,2).


In [ ]:
staged = raw.with_columns([
    pl.col("orders_budget").cast(pl.Int64, strict=False),
    pl.col("net_sales_czk_budget").cast(pl.Decimal(18, 2), strict=False),
])
if staged["orders_budget"].null_count() or staged["net_sales_czk_budget"].null_count():
    raise ValueError("Invalid budget numeric value encountered during parsing")
if staged.schema["orders_budget"] != pl.Int64 or staged.schema["net_sales_czk_budget"] != pl.Decimal(18, 2):
    raise TypeError("Unexpected budget numeric types")
if staged.filter((pl.col("orders_budget") <= 0) | (pl.col("net_sales_czk_budget") < 0)).height:
    raise ValueError("Budget values must be valid non-negative financial values with positive order targets")
if set(staged["shop"].unique().to_list()) != set(EXPECTED_SHOPS):
    raise ValueError("Unexpected budget shop values")


# 4. Data Quality & Reconciliation

Validate unique month-shop combinations, 27 source rows, and three shops for each of the nine months.


In [ ]:
if staged.height != EXPECTED_ROWS or staged.select(["month", "shop"]).n_unique() != EXPECTED_ROWS:
    raise ValueError("Budget row count or month-shop uniqueness validation failed")
shop_month_counts = staged.group_by("month").len()
if shop_month_counts.height != 9 or shop_month_counts.filter(pl.col("len") != 3).height:
    raise ValueError("Expected three shop budgets for each of nine months")
display(shop_month_counts.sort("month"))
display(staged.sort(["month", "shop"]).head(9))


# 5. Parquet Export

Write original CZK budgets without variance calculations or other business transformations.


In [ ]:
staged.write_parquet(OUTPUT_PATH)
exported = pl.read_parquet(OUTPUT_PATH)
if exported.height != EXPECTED_ROWS or exported.select(["month", "shop"]).n_unique() != EXPECTED_ROWS:
    raise ValueError(f"Budget Parquet export validation failed: {OUTPUT_PATH}")
if exported.schema != staged.schema:
    raise TypeError(f"Budget Parquet schema differs from staged schema: {OUTPUT_PATH}")
display(pl.DataFrame({"output_file": [OUTPUT_PATH.name], "rows": [exported.height], "schema": [str(exported.schema)]}))
